# AI4I 2020 — Exploratory Data Analysis

This notebook performs the initial exploratory analysis of the AI4I 2020 Predictive Maintenance Dataset.

The main objectives are:

- Understand the dataset structure.
- Inspect feature types.
- Identify missing and duplicated values.
- Analyze the target variable.
- Detect class imbalance.
- Identify potentially irrelevant features.
- Identify possible sources of data leakage.
- Define the next preprocessing steps.

No transformations will be permanently applied to the dataset in this notebook.

In [1]:
from pathlib import Path

import pandas as pd     # Trabajar con datos estructurados
import numpy as np      # Realizar cálculos numéricos y científicos

In [2]:
DATA_PATH = Path("../data/raw/ai4i2020.csv")

print(DATA_PATH)

df = pd.read_csv(DATA_PATH)

..\data\raw\ai4i2020.csv


In [3]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 1. Dataset Shape

First, we inspect the number of observations and variables available in the dataset.

In [19]:
print(df.shape)

rows, columns = df.shape

print(f"Rows: {rows}")
print(f"Columns: {columns}")

(10000, 14)
Rows: 10000
Columns: 14


## 2. Column Names

We inspect all available variables before deciding which features may be useful for the predictive task.

In [3]:
df.columns.tolist()

['UDI',
 'Product ID',
 'Type',
 'Air temperature [K]',
 'Process temperature [K]',
 'Rotational speed [rpm]',
 'Torque [Nm]',
 'Tool wear [min]',
 'Machine failure',
 'TWF',
 'HDF',
 'PWF',
 'OSF',
 'RNF']

## 3. Dataset Information

The following inspection shows the data types, number of non-null values, and memory usage of each variable.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

## 4. Missing Values

Missing values can affect preprocessing and model training. We inspect whether any feature contains null values.

In [5]:
df.isnull().sum()

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

## 5. Duplicate Records

Duplicated observations may bias model training if the same examples are represented multiple times.

In [7]:
duplicate_count = df.duplicated().sum()

print(f"Duplicated rows: {duplicate_count}")

Duplicated rows: 0


## 6. Descriptive Statistics

Descriptive statistics help us understand the scale, range, central tendency, and variability of numerical features.

In [8]:
df.describe()

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2886.89568,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,1.00000,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,5000.50000,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7500.25000,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,10000.00000,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


## 7. Target Variable

The initial predictive problem is binary classification.

The target variable is:

`Machine failure`

where:

- `0` represents no machine failure.
- `1` represents machine failure.

In [9]:
df["Machine failure"].value_counts()

Machine failure
0    9661
1     339
Name: count, dtype: int64

In [10]:
df["Machine failure"].value_counts(normalize=True)

Machine failure
0    0.9661
1    0.0339
Name: proportion, dtype: float64

In [11]:
df["Machine failure"].value_counts(normalize=True) * 100

Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64

## 8. Product Type Distribution

`Type` is a categorical variable representing the product quality variant.

We inspect its available categories and their distribution.

In [12]:
df["Type"].value_counts()

Type
L    6000
M    2997
H    1003
Name: count, dtype: int64

In [25]:
df["Type"].value_counts(normalize=True) * 100

Type
L    60.00
M    29.97
H    10.03
Name: proportion, dtype: float64

In [26]:
df["Type"].unique()

array(['M', 'L', 'H'], dtype=object)

## 9. Identifier Features

`UDI` and `Product ID` appear to act primarily as identifiers rather than physical measurements.

Identifiers usually do not provide meaningful information about the underlying machine condition and may introduce artificial patterns into the model.

Their usefulness will therefore be evaluated before preprocessing.

In [13]:
df[["UDI", "Product ID"]].head(10)

,UDI,Product ID
0,1,M14860
1,2,L47181
2,3,L47182
3,4,L47183
4,5,L47184
5,6,M14865
6,7,L47186
7,8,L47187
8,9,M14868
9,10,M14869


In [14]:
df["UDI"].nunique()

df["Product ID"].nunique()

10000

In [15]:
len(df)

10000

## 10. Failure Type Variables

The dataset contains additional binary variables representing specific failure mechanisms:

- TWF — Tool Wear Failure
- HDF — Heat Dissipation Failure
- PWF — Power Failure
- OSF — Overstrain Failure
- RNF — Random Failure

These variables require special attention because they may directly describe the event represented by the target variable `Machine failure`.

Using them as model inputs could introduce target leakage.

In [16]:
failure_columns = [
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
]

df[failure_columns].sum()

TWF     46
HDF    115
PWF     95
OSF     98
RNF     19
dtype: int64

In [17]:
df[
    ["Machine failure"] + failure_columns
].head(20)

,Machine failure,TWF,HDF,PWF,OSF,RNF
0,0,0,0,0,0,0
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0
5,0,0,0,0,0,0
6,0,0,0,0,0,0
7,0,0,0,0,0,0
8,0,0,0,0,0,0
9,0,0,0,0,0,0


In [18]:
df.groupby("Machine failure")[failure_columns].sum()

,TWF,HDF,PWF,OSF,RNF
Machine failure,,,,,
0,0,0,0,0,18
1,46,115,95,98,1


In [19]:
df["failure_type_sum"] = df[failure_columns].sum(axis=1)

pd.crosstab(
    df["Machine failure"],
    df["failure_type_sum"]
)

failure_type_sum,0,1,2,3
Machine failure,,,,
0,9643,18,0,0
1,9,306,23,1


In [20]:
df.drop(columns=["failure_type_sum"], inplace=True)

## 11. Numerical Features

We inspect the main physical and operational variables that are potential predictive features.

In [21]:
numerical_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

df[numerical_features].describe().T

,count,mean,std,min,25%,50%,75%,max
Air temperature [K],10000.0,300.00493,2.000259,295.3,298.3,300.1,301.5,304.5
Process temperature [K],10000.0,310.00556,1.483734,305.7,308.8,310.1,311.1,313.8
Rotational speed [rpm],10000.0,1538.77610,179.284096,1168.0,1423.0,1503.0,1612.0,2886.0
Torque [Nm],10000.0,39.98691,9.968934,3.8,33.2,40.1,46.8,76.6
Tool wear [min],10000.0,107.95100,63.654147,0.0,53.0,108.0,162.0,253.0


## 12. Numerical Correlation

Correlation provides an initial indication of linear relationships between numerical variables.

Correlation does not imply causation and should not be used as the only criterion for feature selection.

In [22]:
correlation_matrix = df[numerical_features + ["Machine failure"]].corr()

correlation_matrix

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
Air temperature [K],1.000000,0.876107,0.022670,-0.013778,0.013853,0.082556
Process temperature [K],0.876107,1.000000,0.019277,-0.014061,0.013488,0.035946
Rotational speed [rpm],0.022670,0.019277,1.000000,-0.875027,0.000223,-0.044188
Torque [Nm],-0.013778,-0.014061,-0.875027,1.000000,-0.003093,0.191321
Tool wear [min],0.013853,0.013488,0.000223,-0.003093,1.000000,0.105448
Machine failure,0.082556,0.035946,-0.044188,0.191321,0.105448,1.000000


## 13. Compare successful and failed observations

In [23]:
df.groupby("Machine failure")[numerical_features].mean()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
Machine failure,,,,,
0,299.973999,309.995570,1540.260014,39.629655,106.693717
1,300.886431,310.290265,1496.486726,50.168142,143.781711


In [24]:
df.groupby("Machine failure")[numerical_features].median()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
Machine failure,,,,,
0,300.0,310.0,1507.0,39.9,107.0
1,301.6,310.4,1365.0,53.7,165.0


## 14. Initial Findings

Based on the exploratory analysis:

- The dataset contains 10,000 machine observations.
- The target variable is `Machine failure`.
- The predictive problem is initially defined as binary classification.
- The target variable is highly imbalanced, with machine failures representing a small minority of the observations.
- Accuracy alone will therefore not be sufficient for model evaluation.
- Recall, Precision, F1-score, ROC-AUC, and the confusion matrix will be considered during evaluation.
- `Type` is a categorical variable and will require encoding before being used by most Machine Learning models.
- `UDI` behaves as a unique observation identifier and should not be used as a predictive feature.
- `Product ID` also behaves primarily as an identifier and will likely be excluded from the predictive feature set.
- `TWF`, `HDF`, `PWF`, `OSF`, and `RNF` describe specific failure mechanisms and are strongly related to the target variable.
- These failure-type variables should not be used as input features when predicting `Machine failure`, because doing so could introduce target leakage.
- The primary candidate features for the first predictive models are therefore:

  - `Type`
  - `Air temperature [K]`
  - `Process temperature [K]`
  - `Rotational speed [rpm]`
  - `Torque [Nm]`
  - `Tool wear [min]`

- No permanent transformations have been performed on the raw dataset.

### Class Imbalance Observation

The target variable `Machine failure` is highly imbalanced.

- Class `0` (No Failure): 96.61%
- Class `1` (Failure): 3.39%

This means that accuracy alone will not be a reliable evaluation metric. A model that always predicts `0` could achieve approximately 96.61% accuracy while failing to detect any actual machine failures.

For this reason, special attention will be given to:

- Recall
- Precision
- F1-score
- Confusion Matrix
- ROC-AUC

The positive class (`Machine failure = 1`) will be particularly important because false negatives represent real failures that the model failed to detect.

### Target Leakage Analysis

The failure-type variables `TWF`, `HDF`, `PWF`, `OSF`, and `RNF` are highly related to the target variable `Machine failure`.

Out of the 339 observations labeled as machine failures, 330 contain at least one active failure-type indicator. Therefore, using these variables as predictive features would provide the model with information that directly describes the failure event.

This would introduce target leakage and could produce unrealistically high evaluation metrics without representing the intended predictive maintenance problem.

For this reason, the failure-type variables will be excluded from the input feature set.

The initial predictive features will be:

- `Type`
- `Air temperature [K]`
- `Process temperature [K]`
- `Rotational speed [rpm]`
- `Torque [Nm]`
- `Tool wear [min]`

The target variable will be:

- `Machine failure`

Additionally, `UDI` and `Product ID` will be excluded because they primarily act as record identifiers rather than physical machine-condition variables.

In [28]:
df["Type"].value_counts()

df["Type"].value_counts(normalize=True) * 100

Type
L    60.00
M    29.97
H    10.03
Name: proportion, dtype: float64

In [29]:
df.groupby("Type")["Machine failure"].agg(["count", "sum", "mean"])

,count,sum,mean
Type,,,
H,1003,21,0.020937
L,6000,235,0.039167
M,2997,83,0.027694
